# 04 — Export georeferenced Hakaluki maps and produce local cartography

**Run after notebooks 00 and 01.** This notebook submits optional Google Earth Engine export tasks to Google Drive, and prepares local TIFF-to-PNG map layouts after you manually download completed TIFFs into `F:\Wetland Test\results\rasters`. Exporting is explicitly **opt-in**. Map values come only from Earth Engine or downloaded GeoTIFFs; this notebook creates no synthetic observations.

**Interpretation:** SAR water maps are screening classifications; Dynamic World is cloud-limited habitat screening; TROPOMI XCH₄ is coarse atmospheric concentration, **not surface methane flux**. Vegetation/monthly maps are not direct flood-event duration measurements.

## A. Initialize the configured Cloud project and spatial domain

In [ ]:
from pathlib import Path
import json, sys
import pandas as pd
ROOT = Path(r"F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project")
sys.path.insert(0,str(ROOT))
from src.hakaluki_core import project_paths, load_boundary
from src.ee_pipeline import (initialize, geometry_from_geodataframe, choose_s1_geometry,
    s1_month_water, optical_s2_month, dynamic_world_month,
    start_drive_geotiff_export, CH4_ID, CH4_BAND, CHIRPS_ID, ERA5_ID)
import ee
P=project_paths(ROOT)
(P['root']/'results'/'rasters').mkdir(parents=True,exist_ok=True)
CFG=json.loads((ROOT/'config/settings.json').read_text(encoding='utf-8'))
verified=P['boundary']/'hakaluki_verified.geojson'
candidate=P['boundary']/'hakaluki_analysis_candidate.geojson'
assert verified.exists() or candidate.exists(), 'Run notebook 00 before exporting rasters.'
GDF=load_boundary(verified if verified.exists() else candidate)
initialize(CFG['earth_engine_cloud_project'])
ROI=geometry_from_geodataframe(GDF)
ATMOS=ROI.buffer(CFG['ch4_buffer_m'])
GEOM=choose_s1_geometry(ROI,CFG['pilot_year'])
print('GEE project:', CFG['earth_engine_cloud_project'],'| GIS boundary:', ('verified provided' if verified.exists() else 'UNVERIFIED candidate'))
print('Orbit:',GEOM['orbit_pass'],GEOM['relative_orbit'])

## B. Assemble 2022 exploratory maps

- SAR: March and June median-VV Otsu screening masks, **30 m export scale**.
- Sentinel-2: March and June NDVI / MNDWI, **20 m export scale**.
- Dynamic World: modal annual habitat classification, **30 m export scale**.
- CHIRPS: April–June accumulated rainfall (mm), **5.5 km export scale**.
- ERA5-Land: June average 2 m temperature (°C), **11 km export scale**.
- TROPOMI L3 CH₄: 2022 mean XCH₄, **7 km export scale** for regional presentation. This does **not** confer 7 km independent footprint resolution.

These intervals are working examples; adjust them based on the final hydrograph, flood documentation, and observation support.

In [ ]:
EXPORT_LAYERS={}
for label,start,end in [('SAR_water_Mar2022','2022-03-01','2022-04-01'),('SAR_water_Jun2022','2022-06-01','2022-07-01')]:
    img,meta=s1_month_water(ROI,start,end,GEOM['orbit_pass'],GEOM['relative_orbit'])
    if img is not None:
        EXPORT_LAYERS[label]={'image':img,'region':ROI,'scale':30,'as_byte':True,'units':'unvalidated 0/1 low-VV open-water screen'}
    else: print('No SAR screening image for',label,meta)
for label,start,end in [('Sentinel2_Mar2022','2022-03-01','2022-04-01'),('Sentinel2_Jun2022','2022-06-01','2022-07-01')]:
    img,meta=optical_s2_month(ROI,start,end)
    if img is not None:
        for band in ['NDVI','MNDWI']:
            EXPORT_LAYERS[f'{band}_{label}']={'image':img.select(band),'region':ROI,'scale':20,'as_byte':False,'units':'index'}
    else: print('No usable S2 source collection for',label,meta)
class_img,class_stats=dynamic_world_month(ROI,'2022-01-01','2023-01-01')
if class_img is not None:
    EXPORT_LAYERS['DynamicWorld_modal_2022']={'image':class_img,'region':ROI,'scale':30,'as_byte':True,'units':'categorical DW class code'}
rain=ee.ImageCollection(CHIRPS_ID).filterDate('2022-04-01','2022-07-01').select('precipitation').sum().rename('rainfall_mm')
EXPORT_LAYERS['CHIRPS_rain_AprJun2022']={'image':rain,'region':ROI,'scale':5500,'as_byte':False,'units':'mm over period'}
era=ee.ImageCollection(ERA5_ID).filterDate('2022-06-01','2022-07-01').select('temperature_2m').mean().subtract(273.15).rename('temperature_C')
EXPORT_LAYERS['ERA5Land_temp_Jun2022']={'image':era,'region':ROI,'scale':11000,'as_byte':False,'units':'degrees C'}
ch4col=(ee.ImageCollection(CH4_ID).filterBounds(ATMOS).filterDate('2022-01-01','2023-01-01').select(CH4_BAND))
if ch4col.size().getInfo()>0:
    EXPORT_LAYERS['TROPOMI_XCH4_mean_2022']={'image':ch4col.mean().rename('XCH4_ppb'),'region':ATMOS,'scale':7000,'as_byte':False,'units':'XCH4 ppb; not flux'}
print('Prepared export specifications:',len(EXPORT_LAYERS))
print('No Google Drive tasks have been started yet.')

## C. Opt in to submitting Google Drive export tasks

The next cell is safe by default. Change `SUBMIT_EXPORTS` to `True` to start asynchronous GEE Drive exports. Jobs can take time and may fail with project quota/permission errors. Check status in the EE Tasks interface. Download the GeoTIFFs into `F:\Wetland Test\results\rasters` for local mapping.

The manifest records attempted tasks and expected output filenames, not task completion.

In [ ]:
SUBMIT_EXPORTS=False  # change to True when ready
export_records=[]
for name,spec in EXPORT_LAYERS.items():
    desc=f'HAKALUKI_{name}'
    if SUBMIT_EXPORTS:
        task=start_drive_geotiff_export(spec['image'],spec['region'],desc,
            drive_folder=CFG['ee_export_drive_folder'],scale=spec['scale'],as_byte=spec['as_byte'])
        task_id=task.id
        status=task.status().get('state','UNKNOWN')
        print(desc,status,task_id)
    else:
        task_id=''
        status='NOT_SUBMITTED'
    export_records.append({'name':name,'drive_prefix':desc,'expected_tif':desc+'.tif',
        'requested_scale_m':spec['scale'],'units':spec['units'],
        'task_id':task_id,'status_at_submission':status,
        'boundary_status':CFG['study_area_status']})
manifest=pd.DataFrame(export_records)
manifest.to_csv(P['tables']/'gee_export_manifest.csv',index=False)
display(manifest)

## D. Produce PNG maps from **downloaded real GeoTIFFs**

This plotting section skips files that have not been downloaded. It plots native rasters without adding fictitious detail. The exact matched boundary is converted to each raster's coordinate reference system. Each generated image is saved in `results/maps` and labelled according to its actual variable and scale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import plotting_extent
from matplotlib.colors import ListedColormap
from IPython.display import display, Image

def map_real_raster(tif_file, title, output_png, categorical=False):
    with rasterio.open(tif_file) as src:
        if src.crs is None:
            raise ValueError(f'GeoTIFF missing CRS: {tif_file}')
        layer=src.read(1, masked=True)
        extent=plotting_extent(src)
        g=GDF.to_crs(src.crs)
        fig,ax=plt.subplots(figsize=(9,7))
        if categorical:
            cmap=ListedColormap(['#419BDF','#397D49','#88B053','#7A87C6','#E49635',
                                 '#DFC35A','#C4281B','#A59B8F','#B39FE1'])
            im=ax.imshow(layer,extent=extent,origin='upper',cmap=cmap,vmin=-.5,vmax=8.5,
                         interpolation='nearest')
        else:
            im=ax.imshow(layer,extent=extent,origin='upper',cmap='viridis',interpolation='nearest')
        g.boundary.plot(ax=ax,color='orangered',linewidth=1.3)
        fig.colorbar(im,ax=ax,shrink=.7)
        ax.set(title=title, xlabel=f'Easting/longitude ({src.crs})',ylabel=f'Northing/latitude ({src.crs})')
        ax.text(0.01,0.01,'Exploratory satellite/model-derived layer; field calibration pending',
                transform=ax.transAxes,fontsize=8,bbox={'facecolor':'white','alpha':.88,'edgecolor':'none'})
        fig.tight_layout();fig.savefig(output_png,dpi=200,bbox_inches='tight');plt.close(fig)
    return output_png

generated=[]
for record in export_records:
    src=P['root']/'results'/'rasters'/record['expected_tif']
    if not src.exists():
        print('Awaiting exported GeoTIFF:',src.name)
        continue
    dest=P['maps']/f"map_{record['name']}.png"
    generated.append(map_real_raster(src,record['name']+' ('+record['units']+')',dest,
        categorical=record['name'].startswith('DynamicWorld_')))
print('Actual satellite-derived PNGs produced:',len(generated))
for img in generated[:4]:
    display(Image(filename=str(img),width=620))

## E. Interpretation and next-stage calibration

1. The flood maps and habitat classifications are **not** independently validated yet. Prepare a stratified reference dataset and confusion matrices before publishing area changes.
2. If a methane grid contains few independent retrievals, do not map it as an informative local methane hotspot.
3. For any field sample collected later, preserve plot ID, UTC/local observation time, habitat, methane-flux unit, chamber protocol, instrument, water depth, replicate number, and QA flags.
4. Re-run figures and models when the boundary or classification method is revised; do not silently reuse old monthly checkpoints.

Scientific resources: [Earth Engine export documentation](https://developers.google.com/earth-engine/guides/exporting), [Sentinel-5P CH₄ catalogue](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S5P_OFFL_L3_CH4).